# 05 — TTS: XTTS-v2 Fine-tuning (Kaggle / Colab / Local)

Fine-tunes Coqui XTTS-v2 trên ViMD audio cho dialect-conditioned TTS.

**Target KPIs**: MOS ≥ 4.3 | Speaker Similarity ≥ 0.90

> ⚠️ XTTS-v2 full fine-tuning yêu cầu ~14GB VRAM. T4 (15GB) đủ với batch_size=2.

> **Checkpoint**: Tự động skip nếu model đã export.

In [ ]:
# ── Cell 0: Environment + Paths (Kaggle / Colab / Local) ─────────────────────
import sys, os
from pathlib import Path

if os.path.exists('/kaggle/working/project/training/notebooks'):
    sys.path.insert(0, '/kaggle/working/project/training/notebooks')

from _env import *

TTS_CKPT   = os.path.join(CHECKPOINT_DIR, 'tts')
EXPORT_DIR = os.path.join(WEIGHTS_DIR, 'xtts-v2')
Path(TTS_CKPT).mkdir(parents=True, exist_ok=True)
Path(EXPORT_DIR).mkdir(parents=True, exist_ok=True)

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'

if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'ENV    : {ENV_NAME}')
    print(f'Device : {device} | GPU: {torch.cuda.get_device_name(0)} | VRAM: {vram_gb:.1f}GB')
    if vram_gb < 14:
        print('⚠️  VRAM < 14GB — sẽ dùng batch_size=1')
    vram_gb_for_config = vram_gb
else:
    print(f'ENV    : {ENV_NAME}')
    print('⚠️  Không có GPU — TTS fine-tuning sẽ rất chậm')
    vram_gb_for_config = 0

print(f'AUDIO_DIR  : {AUDIO_DIR}')
print(f'TTS_CKPT   : {TTS_CKPT}')
print(f'EXPORT_DIR : {EXPORT_DIR}')

CKPT_DONE = (os.path.exists(os.path.join(EXPORT_DIR, 'config.json')) or
             os.path.exists(os.path.join(EXPORT_DIR, 'model.pth')))
if CKPT_DONE:
    print(f'\n✅ CHECKPOINT: TTS model đã export')
else:
    print('\n⏳ Chưa có TTS model — sẽ train...')

if IS_KAGGLE:
    print('\n⚠️  Kaggle: Nhớ Save Version sau khi training xong!')

In [ ]:
# ── Cell 1: Cài TTS ──────────────────────────────────────────────────────────
import subprocess
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'TTS', 'mlflow'],
                   capture_output=True, text=True)
if r.returncode == 0:
    print('✓ TTS installed')
else:
    print(f'⚠️  TTS install error:')
    print(r.stderr[-300:])

In [ ]:
# ── Cell 2: Kiểm tra VRAM + chuẩn bị metadata ────────────────────────────────
import pandas as pd, glob

feat_files = sorted(glob.glob(os.path.join(FEATURES_DIR, 'feat_batch_*.parquet')))

if feat_files:
    df = pd.concat([pd.read_parquet(p) for p in feat_files[:3]], ignore_index=True)
    print(f'Loaded {len(df):,} samples (3 batches)')
else:
    if os.path.isdir(AUDIO_DIR):
        wav_files = [f for f in os.listdir(AUDIO_DIR) if f.endswith('.wav')][:100]
        df = pd.DataFrame({'filename': wav_files, 'text': ['sample'] * len(wav_files)})
        print(f'⚠️  Feature parquet không có. Dùng {len(df)} audio files')
    else:
        raise RuntimeError('Không tìm thấy features. Chạy 01_dataset_prep trước.')

audio_col = 'filename'
text_col  = 'text' if 'text' in df.columns else 'transcript'
prov_col  = 'province' if 'province' in df.columns else 'region'

xtts_meta = pd.DataFrame({
    'audio_file'  : df[audio_col].apply(lambda fn: os.path.join(AUDIO_DIR, fn)),
    'text'        : df[text_col],
    'speaker_name': df[prov_col] if prov_col in df.columns else 'unknown',
})
xtts_meta = xtts_meta[xtts_meta['audio_file'].apply(os.path.exists)].reset_index(drop=True)

meta_path = os.path.join(TTS_CKPT, 'metadata.csv')
xtts_meta.to_csv(meta_path, sep='|', index=False, header=False)
print(f'XTTS metadata: {len(xtts_meta):,} utterances → {meta_path}')

In [ ]:
# ── Cell 3: XTTS-v2 Fine-tuning ───────────────────────────────────────────────
if CKPT_DONE:
    print('⏭️  Skip training')
elif device == 'cpu':
    print('⚠️  Bỏ qua (không có GPU). Chạy Cell 4 để test inference.')
else:
    try:
        from TTS.config.shared_configs import BaseDatasetConfig
        from TTS.tts.configs.xtts_config import XttsConfig
        from TTS.tts.models.xtts import XttsArgs, XttsAudioConfig, Xtts
        from TTS.utils.trainer import Trainer, TrainerArgs
    except ImportError as e:
        print(f'❌ TTS import lỗi: {e}'); raise

    batch = 1 if vram_gb_for_config < 14 else 2

    dataset_config = BaseDatasetConfig(
        formatter='ljspeech',
        meta_file_train='metadata.csv',
        path=TTS_CKPT,
        language='vi',
    )
    audio_config = XttsAudioConfig(
        sample_rate=22050, dvae_sample_rate=22050, output_sample_rate=24000,
    )
    config = XttsConfig(
        output_path=TTS_CKPT,
        model_args=XttsArgs(use_dvae_training=True),
        audio=audio_config,
        run_name='vietspeech-xtts',
        project_name='DisaTravel_AI',
        dashboard_logger='mlflow',
        logger_uri=os.path.join(LOGS_DIR, 'mlflow'),
        batch_size=batch,
        eval_batch_size=1,
        num_loader_workers=NUM_WORKERS,
        save_step=500,
        epochs=50,
        datasets=[dataset_config],
    )

    model_xtts = Xtts.init_from_config(config)
    model_xtts.load_checkpoint(config, checkpoint_dir=None, eval=False)
    model_xtts.cuda()

    trainer = Trainer(
        TrainerArgs(restore_path=None, use_accelerate=False, mixed_precision='fp16'),
        config, output_path=TTS_CKPT,
        model=model_xtts, train_samples=xtts_meta.to_dict('records'),
    )
    trainer.fit()
    print('\n✅ TTS Training complete!')

    if IS_KAGGLE:
        print('⚠️  Kaggle: Nhấn Save Version!')

In [ ]:
# ── Cell 4: Export model ──────────────────────────────────────────────────────
import shutil, glob as _glob

if CKPT_DONE:
    print(f'✅ Model đã ở {EXPORT_DIR}')
else:
    best_ckpts = sorted(_glob.glob(os.path.join(TTS_CKPT, 'best_model*.pth')))
    if best_ckpts:
        shutil.copy(best_ckpts[-1], os.path.join(EXPORT_DIR, 'model.pth'))
        config_src = os.path.join(TTS_CKPT, 'config.json')
        if os.path.exists(config_src):
            shutil.copy(config_src, os.path.join(EXPORT_DIR, 'config.json'))
        print(f'✅ TTS model exported → {EXPORT_DIR}')
    else:
        print('⚠️  Không tìm thấy best checkpoint — dùng base XTTS-v2 cho inference')

if IS_KAGGLE:
    print('\n⚠️  Kaggle: Nhấn Save Version để lưu model!')

## ✅ XTTS-v2 TTS Setup Complete

| Platform | Model location |
|----------|---------------|
| Kaggle | `/kaggle/working/storage/weights/xtts-v2/` |
| Colab | `Drive/GenAI_ViMD/storage/weights/xtts-v2/` |

**Next**: Run `06_voice_clone.ipynb`